# Optimal control of a single spin by using GOAT over GRAPE

In this introductory example we compute the gradients of analytic pulse shapes in [GOAT](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.120.150401) by using the gradients of the time evolition from [GRAPE](10.1016/j.jmr.2004.11.004).
This is performed by using chain rule - 
$$\frac{\partial J}{\partial p} = \sum_k \frac{\partial J}{\partial c_k} \frac{\partial c_k}{\partial p}$$
where $c_k = c(t_k)$ the 'pixelated' control pulse, $\vec{p}$ are the analytical parameters of the control pulse $c(t) \equiv c(\vec{p}, t) $, and $\frac{\partial J}{\partial c_k}$ are the gradients from GRAPE.

## 1. Generate a PWC pulse shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from paraqeet.quantity import Quantity

from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.signal.envelopes import DCRABEnvelope
from paraqeet.signal.waveform import FlatTopGaussianFilter

In [ ]:
t_final = 20e-9
tlist = np.linspace(0, t_final, 40)
eps = 5 * np.pi * 1.0  # initial amplitude of the resonator (in MHz)
eps_max = 10 * eps  # maximum amplitude of the resonator (in MHz)

tone = DCRABEnvelope(
    num_components=2,
    max_frequency=5 * 2 * np.pi,
    amplitude=Quantity(eps * 1e6, -eps_max * 1e6, eps_max * 1e6, name="Amplitude"),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final"),
    seed=18537,
)
smooth_tone = FlatTopGaussianFilter(
    envelopes=tone, t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final, name="t_final")
)

gen = PWCGenerator(envelopes=[smooth_tone], tlist=tlist, max_amplitude=np.sqrt(2) * eps_max * 1e6)

In [ ]:
params = gen.get_parameters()

In [ ]:
ts = np.linspace(0, t_final, 501)

tone_shape = smooth_tone.compute_output(ts)

plt.plot(ts / 1e-9, np.real(tone_shape) / 1e6, label="Smooth curve real")
plt.plot(ts / 1e-9, np.imag(tone_shape) / 1e6, label="Smooth curve imag")
plt.plot(ts / 1e-9, np.real(gen.generate_signal(ts)) / 1e6, ls="--", label="in-phase")
plt.plot(
    ts / 1e-9,
    np.imag(gen.generate_signal(ts)) / 1e6,
    ls="--",
    label="out-of-phase",
)

plt.xlabel("Time [in ns]")
plt.ylabel("Amplitude [in MHz]")
plt.legend()
plt.show()

## 2. Define Hamiltonian in the rotating frame of drive

As a simple toy model, we use a single spin.

In [ ]:
from paraqeet.model.closed_system import ClosedSystem
from paraqeet.model.rotating_frame_drive import RotatingFrameDrive
from paraqeet.model.hamiltonian import Hamiltonian


class SpinRWA(Hamiltonian):
    """A Single Spin."""

    def __init__(self, drives=None):
        super().__init__(drives)
        self.sigma_p = np.array([[0j, 1], [0, 0]])
        self.dim = 2

    def get_matrix_one_time(self, t):
        """Just sigma-X."""
        return self._drives[0].get_matrix_one_time(self.sigma_p, t)

    def gradient(self, t):
        """Gradient is just the drive matrix."""
        return self._drives[0].gradient(self.sigma_p, t)


drive = RotatingFrameDrive(gen)
spin = SpinRWA(drives=[drive])
model = ClosedSystem(spin)

Using GRAPE as the method to propagate and compute the gradients

In [ ]:
from paraqeet.measurement.state_transfer_fidelity import StateTransferFidelityGRAPE
from paraqeet.propagation.scipy_expm_grape import ScipyExpmGRAPE


prop = ScipyExpmGRAPE(model, res=1e9)

init = np.array([[1.0], [0]])  # |0>
target = np.array([[0.0], [1]])  # |1>

prop.set_initial_state(init)
prop.target_state = target

zeroone = StateTransferFidelityGRAPE(
    propagation=prop,
    initial_state=init,
    target_state=target,
    times=tlist,
)

In [ ]:
def plot_states(propagation, generator):
    """Plot the states."""
    ts = np.linspace(0, t_final, 101)
    states = propagation.propagate(ts)
    sig = generator.generate_signal(ts)

    _, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, np.real(sig), label="I")
    ax[0].plot(ts / 1e-9, np.imag(sig), label="Q")
    ax[0].legend(loc=1)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states)[:, :, 0] ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    plt.show()


plot_states(prop, gen)

## 3. Optimisation
Finally, we define the `GOATOverGRAPE` fideltiy that chains together the GRAPE gradients to compute the gradient wrt the tone parameters

In [ ]:
params = tone.get_parameters()
params

In [ ]:
from paraqeet.optimisation_map import OptimisationMap
from paraqeet.optimisers.dcrab_optimiser_gradient import DCRABOptimiserGradient
from paraqeet.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient
from paraqeet.measurement.goat_over_grape import GOATOverGRAPE

optmap = OptimisationMap()
optmap.add(tone, [params[0]] + params[2:])
optmap.register_params_with_optimisables()

goat = GOATOverGRAPE(zeroone, generators=[gen], generators_order=[0])
opt_grad = DCRABOptimiserGradient(
    goat,
    optimisation_map=optmap,
    super_iteration_every=150,
    max_super_iteration_num=5,
    print_every_iteration_num=10,
    super_iteration_tol=1e-9,
)

In [ ]:
optmap

In [ ]:
opt_grad.optimise()

For this simple example, we reach near perfect fidelity within a few iterations.

In [ ]:
plot_states(prop, gen)

In [ ]:
optmap

In [ ]:
gen.multiply_flat_top = True

opt_map_grape = OptimisationMap()
opt_map_grape.add(gen)
opt_map_grape

In [ ]:
opt_grape = ScipyOptimiserGradient(zeroone, opt_map_grape)

In [ ]:
opt_grape.optimise()

In [ ]:
plot_states(prop, gen)

In [ ]:
opt_map_grape